Here's a more engaging, conversational rewrite of your introduction:

---

# LLM as a Judge: Evaluating Agent Responses

This notebook introduces **LLM as a Judge**, a powerful evaluation technique that leverages language models to assess the quality of agent outputs—essentially using AI to evaluate AI.

## The Challenge

When it comes to evaluating agent responses, traditional metrics like ROUGE scores and exact string matching simply don't cut it. These conventional approaches struggle with the nuances of natural language: they can't tell that "NYC" and "New York City" mean the same thing, they have no way to judge whether a response is actually helpful or appropriate, and they completely miss the context and intent behind user queries. Perhaps most critically, they fail to evaluate the quality of multi-step reasoning—a fundamental capability of modern AI agents.

## The Solution: LLM as a Judge

What if we could use the very capabilities that make language models powerful—their semantic understanding, contextual awareness, and reasoning abilities—to evaluate agent outputs? That's exactly what LLM as a Judge does. By using language models as evaluators, we gain:

- **Semantic understanding** that goes far beyond surface-level text matching
- **Context-aware evaluation** that considers the full picture of each interaction
- **Natural language explanations** that help us understand why a response succeeds or falls short
- **Flexibility** to adapt our evaluation criteria on the fly, without retraining models

## What You'll Learn

This notebook will guide you through the complete process of implementing LLM as a Judge evaluation. You'll start by understanding the core concepts and identifying the right use cases, then build a test agent to evaluate. From there, you'll learn how to design effective judge prompts that elicit meaningful assessments, implement a single judge evaluation system, and analyze the results to gain actionable insights into your agent's performance.

## Prerequisites

Before diving in, you should have:
- A basic understanding of AI agents (check out our [Function Calling Agent](../Function_Calling/Function_Calling_Agent.ipynb) notebook if you need a refresher)
- Familiarity with agent testing concepts (see [testing_agents.md](../../testing_agents.md))
- Access to watsonx.ai with Granite models

## Next Steps

Once you've mastered the basics of LLM as a Judge, you'll be ready to explore more advanced techniques:
- **[Ensemble Judges](Ensemble_Judges.ipynb)** – Learn how using multiple judges can provide more reliable and robust evaluation

# Part 1: Understanding LLM as a Judge

## What is LLM as a Judge?

LLM as a Judge is an evaluation technique where a language model is used to assess the quality of another model's outputs. Rather than forcing responses through the narrow lens of rigid metrics, we tap into the sophisticated language understanding capabilities of LLMs to make nuanced, context-aware judgments about quality.

### Key Concepts

**Meta-Evaluation** is the practice of using an LLM to evaluate another LLM's output. Think of it as having an expert reviewer who can understand the full context of an interaction. The judge model needs to grasp several critical elements: the user's original intent, the surrounding context (including tool results and conversation history), what actually constitutes a "good" response in this specific situation, and how to score or categorize that quality in a meaningful way.

**Evaluation Dimensions** represent the multiple lenses through which judge models can assess responses:
- **Accuracy**: Does the response correctly interpret and use tool results?
- **Completeness**: Does it address all parts of the query, or leave gaps?
- **Helpfulness**: Is it actionable, clear, and genuinely useful to the user?
- **Relevance**: Does it stay focused on the topic at hand?
- **Safety**: Does it avoid harmful, biased, or inappropriate content?

### Why Use LLM as a Judge?

Traditional metrics fall short in four fundamental ways:

**Semantic Equivalence** is invisible to conventional metrics. Consider "The weather in NYC is 72°F" versus "New York City's temperature is 72 degrees"—these statements are semantically identical, yet they'd score poorly on ROUGE metrics due to different word choices.

**Context Dependency** means that whether a response is "good" isn't absolute—it depends entirely on user intent, available tool results, and the conversation's history. A technically accurate response might still miss the mark if it doesn't address what the user actually needs.

**Multi-dimensional Quality** reveals that responses can succeed on some dimensions while failing on others. A response might be perfectly accurate but unhelpful, or wonderfully complete yet irrelevant to the actual question asked.

**Natural Language Flexibility** means agents can phrase the same information in countless valid ways. Traditional metrics can't recognize this variety, penalizing perfectly good responses simply because they don't match expected patterns.

### When to Use LLM as a Judge

✅ **This approach excels when you're:**
- Evaluating the quality of natural language responses where nuance matters
- Assessing subjective qualities like accuracy, completeness, and helpfulness
- Checking whether responses appropriately leverage tool results
- Working in development and testing phases where you need detailed feedback

❌ **Consider alternatives when you need:**
- Structured tool call validation (programmatic checks are faster and more reliable)
- High-frequency, low-latency evaluation in production systems
- Deterministic, reproducible results every time
- To minimize costs in extremely budget-constrained scenarios

### Limitations

While powerful, LLM as a Judge comes with important trade-offs to consider. 
- **Cost** adds up since each evaluation requires a full LLM inference call. 
- **Latency** is inherently higher than programmatic validation methods. 
- **Non-determinism** means the same input might receive slightly different scores across evaluations. 
- **Judge bias** reminds us that the evaluator LLM has its own limitations and potential blind spots. 
- Finally, **prompt sensitivity** means your results depend heavily on how well you design your evaluation prompts—garbage in, garbage out still applies.

# Part 2: Setup and Installation

In [ ]:
# Install dependencies
! echo "::group::Install Dependencies"
%pip install uv
! uv pip install "git+https://github.com/ibm-granite-community/utils.git" \
    langgraph \
    langchain \
    langchain_ibm
! echo "::endgroup::"

In [ ]:
# Import required libraries
import json
import time
import requests
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Annotated, TypedDict

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.utils.utils import convert_to_secret_str
from langchain_core.messages import HumanMessage, AnyMessage, AIMessage
from langgraph.graph.state import CompiledStateGraph
from langgraph.graph.message import add_messages

from ibm_granite_community.notebook_utils import get_env_var

print("✓ Libraries imported successfully")

### Configure watsonx.ai Access

See [Getting Started with IBM watsonx](https://github.com/ibm-granite-community/granite-kitchen/blob/main/recipes/Getting_Started/Getting_Started_with_WatsonX.ipynb) for setup instructions.

Required environment variables:
- `WATSONX_URL`
- `WATSONX_APIKEY`
- `WATSONX_PROJECT_ID`

Store these in a `.env` file.

In [ ]:
# Initialize the Granite model for our agent
model = "ibm/granite-4-h-small"

model_parameters = {
    "temperature": 0,
    "max_completion_tokens": 200,
    "repetition_penalty": 1.05,
}

llm_agent = init_chat_model(
    model=model,
    model_provider="ibm",
    url=convert_to_secret_str(get_env_var("WATSONX_URL")),
    apikey=convert_to_secret_str(get_env_var("WATSONX_APIKEY")),
    project_id=get_env_var("WATSONX_PROJECT_ID"),
    params=model_parameters,
)

print(f"✓ Agent model initialized: {model}")

# Part 3: Building a Test Agent

Before we can evaluate responses, we need an agent to evaluate. Let's create a simple function-calling agent.

## Define Agent Tools

In [ ]:
# Optional: Get API keys for real data
# AV_STOCK_API_KEY from https://www.alphavantage.co/support/#api-key
# WEATHER_API_KEY from https://openweathermap.org/api
AV_STOCK_API_KEY = convert_to_secret_str(get_env_var("AV_STOCK_API_KEY", "unset"))
WEATHER_API_KEY = convert_to_secret_str(get_env_var("WEATHER_API_KEY", "unset"))

In [ ]:
def get_stock_price(ticker: str, date: str) -> dict:
    """
    Retrieves the lowest and highest stock prices for a given ticker and date.

    Args:
        ticker: The stock ticker symbol, for example, "IBM".
        date: The date in "YYYY-MM-DD" format.

    Returns:
        A dictionary containing the low and high stock prices.
    """
    print(f"🔧 Tool: get_stock_price(ticker={ticker}, date={date})")

    apikey = AV_STOCK_API_KEY.get_secret_value()
    if apikey == "unset":
        print("   Using mock data")
        return {"low": "245.45", "high": "249.03"}

    try:
        url = f"https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={ticker}&apikey={apikey}"
        data = requests.get(url).json()
        return {
            "low": data["Time Series (Daily)"][date]["3. low"],
            "high": data["Time Series (Daily)"][date]["2. high"]
        }
    except Exception as e:
        return {"low": "none", "high": "none"}


def get_current_weather(location: str) -> dict:
    """
    Fetches the current weather for a given location.

    Args:
        location: The city name.

    Returns:
        Weather information including temperature, description, and humidity.
    """
    print(f"🔧 Tool: get_current_weather(location={location})")
    
    apikey = WEATHER_API_KEY.get_secret_value()
    if apikey == "unset":
        print("   Using mock data")
        return {"description": "partly cloudy", "temperature": 22.5, "humidity": 65}

    try:
        url = f"https://api.openweathermap.org/data/2.5/weather?q={location}&appid={apikey}&units=metric"
        data = requests.get(url).json()
        return {
            "description": data["weather"][0]["description"],
            "temperature": data["main"]["temp"],
            "humidity": data["main"]["humidity"]
        }
    except Exception as e:
        return {"description": "none", "temperature": "none", "humidity": "none"}

## Create the Agent

In [ ]:
# Define agent state
class State(TypedDict, total=False):
    """Agent state that holds conversation messages."""
    messages: Annotated[list[AnyMessage], add_messages]

# Create the agent
tools = [get_stock_price, get_current_weather]
agent: CompiledStateGraph = create_agent(model=llm_agent, tools=tools)

print("✓ Agent created with tools:", [t.__name__ for t in tools])

## Test the Agent

In [ ]:
def run_agent(graph: CompiledStateGraph, user_input: str) -> str:
    """Run the agent and return the final response."""
    print(f"\n{'='*60}")
    print(f"👤 User: {user_input}")
    print(f"{'='*60}")
    
    user_message = HumanMessage(user_input)
    input_state = State(messages=[user_message])
    
    final_response = ""
    for event in graph.stream(input_state):
        for value in event.values():
            last_message = value["messages"][-1]
            if isinstance(last_message, AIMessage) and not last_message.tool_calls:
                final_response = last_message.content
    
    print(f"\n🤖 Agent: {final_response}")
    print(f"{'='*60}\n")
    return final_response

# Test the agent
test_response = run_agent(agent, "What is the weather in Miami?")

# Part 4: Implementing LLM as a Judge

Now let's build an evaluation system using LLM as a Judge.

## Initialize the Judge Model

In [ ]:
# Initialize judge model (can be same or different from agent model)
judge_parameters = {
    "temperature": 0,  
    "max_completion_tokens": 500,
    "repetition_penalty": 1.05,
}

llm_judge = init_chat_model(
    model="meta-llama/llama-3-3-70b-instruct",
    model_provider="ibm",
    url=convert_to_secret_str(get_env_var("WATSONX_URL")),
    apikey=convert_to_secret_str(get_env_var("WATSONX_APIKEY")),
    project_id=get_env_var("WATSONX_PROJECT_ID"),
    params=judge_parameters,
)

print("✓ Judge model initialized")

## Design the Judge Prompt

A good judge prompt should:
1. Provide clear evaluation criteria
2. Include all necessary context
3. Request step-by-step reasoning
4. Use consistent scoring format
5. Define what "good" looks like

In [ ]:
def create_judge_prompt(user_query: str, agent_response: str, expected_response: Optional[str] = None, tool_results: Optional[str] = None) -> str:
    """
    Create a comprehensive judge prompt for evaluating agent responses.
    """
    prompt = f"""You are an impartial judge evaluator assessing the quality of an AI agent's response. Assign the agent's reponse a score from 0 to 5 based on the criteria below.

**User Query:**
{user_query}

**Agent Response:**
{agent_response}
"""
    
    if expected_response:
        prompt += f"""
**Ground Truth Text:**
{expected_response}
"""

    if tool_results:
        prompt += f"""
**Tool Results Available:**
{tool_results}
"""
    
    prompt += """
**Evaluation Criteria:**
Evaluate the response based on the following criteria (0-5 scale):

1. **Accuracy**: 
    - The agent answer must align with the ground truth when available.
    - The agent answer must make use of the tool results when available.
    - **Note**: If the response is inaccurate, completeness and relevance are not applicable.
2. **Completeness**: The response must be complete (only evaluated if accurate).
3. **Relevance**: The response must be relevant to the user query (only evaluated if accurate).


**Return Your Rating**
Return your rating in the following format:
{"justification": your_justification, "score": your_score}
"""
    return prompt

## Create Evaluation Data Structures

In [ ]:
@dataclass
class EvaluationScores:
    """Structured scores from judge evaluation."""
    score: int
    justification: str
    raw_response: str
    
    def to_dict(self) -> Dict[str, Any]:
        return {
            "score": self.score,
            "justification": self.justification
        }


@dataclass
class EvaluationResult:
    """Complete evaluation result for a test case."""
    user_query: str
    agent_response: str
    tool_results: Optional[str]
    scores: EvaluationScores
    evaluation_time: float
    
    def to_dict(self) -> Dict[str, Any]:
        return {
            "user_query": self.user_query,
            "agent_response": self.agent_response,
            "tool_results": self.tool_results,
            "scores": self.scores.to_dict(),
            "evaluation_time": self.evaluation_time
        }

print("✓ Data structures defined")

## Implement the Judge Evaluator

In [ ]:
def parse_judge_response(response: str) -> EvaluationScores:
    """
    Parse the judge's response to extract scores.
    
    Note: This is a simple parser. In production, consider more robust parsing
    or structured output formats.
    """
    try:
        _response = json.loads(response)
    except:
        _response = {"score": -1, "justification": "Unable to parse response"}
    
    # Provide defaults if parsing failed
    return EvaluationScores(
        score=_response.get('score', -1),
        justification=_response.get("justification", "Unable to parse response"),
        raw_response=response
    )


def evaluate_with_judge(
    user_query: str,
    agent_response: str,
    expected_response: Optional[str]=None,
    tool_results: Optional[str] = None,
    judge_model = None
) -> EvaluationResult:
    """
    Evaluate an agent response using an LLM judge.
    """
    if judge_model is None:
        judge_model = llm_judge
    
    # Create evaluation prompt
    prompt = create_judge_prompt(user_query, agent_response, expected_response, tool_results)
    
    # Get judge evaluation
    start_time = time.time()
    judge_response = judge_model.invoke(prompt)
    evaluation_time = time.time() - start_time
    
    # Parse response
    response_text = judge_response.content if hasattr(judge_response, 'content') else str(judge_response)
    scores = parse_judge_response(response_text)
    
    return EvaluationResult(
        user_query=user_query,
        agent_response=agent_response,
        tool_results=tool_results,
        scores=scores,
        evaluation_time=evaluation_time
    )

print("✓ Judge evaluator implemented")

# Part 5: Running Evaluations

Let's evaluate some agent responses!

## Example 1: Weather Query

In [ ]:
# Get agent response
query1 = "What is the weather in Miami?"
response1 = run_agent(agent, query1)

# Evaluate it
eval1 = evaluate_with_judge(
    user_query=query1,
    agent_response=response1,
    tool_results="Called get_current_weather(location='Miami') -> {description: 'partly cloudy', temperature: 22.5, humidity: 65}"
)

# Display results
print("\n" + "="*60)
print("EVALUATION RESULTS")
print("="*60)
print(f"\nScores:")
print(f"  Overall:      {eval1.scores.score:.1f}/5")
print(f"\nJustification: {eval1.scores.justification}")
print(f"\nEvaluation time: {eval1.evaluation_time:.2f}s")
print("="*60)

## Example 2: Stock Price Query

In [ ]:
query2 = "What was IBM's stock price on 2024-01-15?"
response2 = run_agent(agent, query2)

eval2 = evaluate_with_judge(
    user_query=query2,
    agent_response=response2,
    tool_results="Called get_stock_price(ticker='IBM', date='2024-01-15') -> {low: '245.45', high: '249.03'}"
)

print("\n" + "="*60)
print("EVALUATION RESULTS")
print("="*60)
print(f"\nScores:")
print(f"  Overall:      {eval2.scores.score:.1f}/5")
print(f"\nJustification: {eval2.scores.justification}")
print(f"\nEvaluation time: {eval2.evaluation_time:.2f}s")
print("="*60)

## Example 3: Invalid Query

In [ ]:
query3 = "What is the current weather in Amsterdam?"
response3 = run_agent(agent, query3)

eval3 = evaluate_with_judge(
    user_query=query3,
    agent_response=response3,
    # expected_response="It is currently 35 degrees Fahrenheit in Amsterdam with a 50% chance of rain.",
    tool_results="No tools called (ambiguous query)"
)

print("\n" + "="*60)
print("EVALUATION RESULTS")
print("="*60)
print(f"\nScores:")
print(f"  Overall:      {eval3.scores.score:.1f}/5")
print(f"\nJustification: {eval3.scores.justification}")
print(f"\nEvaluation time: {eval3.evaluation_time:.2f}s")
print("="*60)

# Part 6: Batch Evaluation

Let's evaluate multiple test cases and analyze the results.

In [ ]:
# Define test cases
test_cases = [
    {
        "query": "What is the weather in Miami?",
        "expected_tool": "get_current_weather",
        "description": "Simple weather query"
    },
    {
        "query": "Get me IBM stock price for 2024-01-15",
        "expected_tool": "get_stock_price",
        "description": "Stock price with date"
    },
    {
        "query": "What's the weather like in NYC?",
        "expected_tool": "get_current_weather",
        "description": "Weather with abbreviation"
    },
    {
        "query": "Tell me about Apple",
        "expected_tool": "clarification",
        "description": "Ambiguous query"
    }
]

# Run evaluations
results = []
for i, test in enumerate(test_cases, 1):
    print(f"\n{'='*60}")
    print(f"Test Case {i}/{len(test_cases)}: {test['description']}")
    print(f"{'='*60}")
    
    # Get agent response
    response = run_agent(agent, test['query'])
    
    # Evaluate
    evaluation = evaluate_with_judge(
        user_query=test['query'],
        agent_response=response,
        tool_results=f"Expected tool: {test['expected_tool']}"
    )
    
    results.append({
        "test_case": test['description'],
        "query": test['query'],
        "evaluation": evaluation
    })
    
    print(f"\n📊 Scores: Score={evaluation.scores.score:.1f}")

print(f"\n✓ Completed {len(results)} evaluations")

# Conclusion

Throughout this notebook, you've built a comprehensive understanding of LLM as a Judge evaluation. You've learned what LLM as a Judge is and developed the judgment to know when it's the right tool for your evaluation needs. You've mastered the art of designing effective judge prompts that provide clear criteria and elicit meaningful assessments. You've implemented single judge evaluation systems using Watsonx.ai models, gaining hands-on experience with the complete evaluation pipeline. 

## Key Takeaways

LLM as a Judge represents a fundamental shift in how we evaluate AI agents—moving from rigid, surface-level metrics to nuanced, context-aware assessment. The quality of your evaluations hinges on your judge prompts: the best ones are specific, well-structured, and explicitly request step-by-step reasoning. While single judge evaluation proves invaluable during development and testing phases, batch evaluation becomes your window into patterns and trends that individual assessments might miss.

## Limitations to Consider

As powerful as this approach is, it's important to understand its constraints. **Cost** accumulates with each evaluation since every assessment requires a full LLM inference call. **Latency** will always be higher than programmatic checks—there's no way around the time needed for model inference. **Variability** is inherent to the approach; the same input may receive slightly different scores across evaluations due to the non-deterministic nature of LLMs. And **judge bias** reminds us that the evaluator itself has limitations and blind spots that can influence assessments.

## Next Steps

You're now ready to explore more advanced evaluation techniques:

**[Ensemble Judges](Ensemble_Judges.ipynb)** will show you how to use multiple judges together, creating more reliable evaluations through consensus and reducing the impact of individual judge biases.

## Additional Resources

Deepen your understanding with these related materials:
- [Testing Agents Guide](../../testing_agents.md) – Comprehensive strategies for agent testing
- [Function Calling Agent](../Function_Calling/Function_Calling_Agent.ipynb) – Build the foundation for evaluable agents
- [Agent TDD](../Agent_TDD/Function_Calling_Agent_TDD.ipynb) – Test-driven development for AI agents
